# Ensemble DQN Training - IMC Prosperity 4

**What this notebook does:**
1. Installs dependencies (stable-baselines3, gymnasium)
2. Uploads/mounts your trading data
3. Computes 19 trading indicators from order book data
4. Trains an Ensemble of N independent DQN agents with different seeds
5. Evaluates by averaging Q-values across members, exports weights for submission

**Run this on:** Google Colab (free GPU) or Kaggle Notebooks

---

## 1. Setup & Install Dependencies

In [ ]:
!pip install stable-baselines3 gymnasium pandas numpy matplotlib --quiet
print("Dependencies installed!")

## 2. Setup Paths

In [ ]:
import os, sys

# Notebook is at RLM/ensemble_dqn/train.ipynb => repo root is ../..
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

## 3. Load Data & Check GPU

In [ ]:
import torch
import numpy as np
import pandas as pd
import os

# Check GPU availability
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("No GPU detected - training on CPU (still fast for this model)")

# Load data
from RLM.shared.data_loader import load_prices, load_trades

prices_df = load_prices(data_dir=DATA_DIR)
trades_df = load_trades(data_dir=DATA_DIR)

print(f"\nPrices: {len(prices_df)} rows")
print(f"Trades: {len(trades_df)} rows")
print(f"Products: {prices_df['product'].unique()}")
print(f"Days: {sorted(prices_df['day'].unique())}")

## 4. Compute Feature Normalization

Before training, we need to compute the mean and standard deviation of each feature across the training data. This ensures all 19 features are on the same scale.

In [ ]:
from RLM.shared.config import PRODUCTS, TRAIN_CONFIG, DQN_CONFIG, NETWORK_CONFIG, ENV_CONFIG
from RLM.shared.features import FeatureComputer, compute_features_from_row, fit_normalizer
from RLM.shared.env import TradingEnv

# Compute normalization parameters from training data
def compute_normalization_params(prices_df, trades_df, products, days):
    all_features = {p: [] for p in products}
    for day in days:
        for product in products:
            fc = FeatureComputer(product)
            day_prices = prices_df[(prices_df["day"] == day) & (prices_df["product"] == product)]
            day_prices = day_prices.sort_values("timestamp").reset_index(drop=True)
            day_trades = trades_df[trades_df["symbol"] == product].sort_values("timestamp")
            for _, row in day_prices.iterrows():
                ts = row["timestamp"]
                ts_trades = day_trades[day_trades["timestamp"] == ts]
                trades = list(zip(ts_trades["price"], ts_trades["quantity"])) if len(ts_trades) > 0 else None
                features = compute_features_from_row(row, fc, position=0, trades=trades)
                all_features[product].append(features)
    combined = np.vstack([np.array(v) for v in all_features.values()])
    return fit_normalizer(combined)

feat_means, feat_stds = compute_normalization_params(
    prices_df, trades_df, PRODUCTS, TRAIN_CONFIG["train_days"]
)

print("Feature normalization computed!")
print(f"Means (first 5): {feat_means[:5]}")
print(f"Stds  (first 5): {feat_stds[:5]}")

## 5. Train Ensemble of DQN Agents

Trains N independent Double DQN agents with different random seeds. At inference, Q-values are averaged across all members for more stable decisions.

**Adjust `TOTAL_TIMESTEPS` and `N_MEMBERS` below.**

In [ ]:
# ============================================================
# TRAINING CONFIG - Adjust these!
# ============================================================
TOTAL_TIMESTEPS = 100_000   # Per member. Increase for better results
N_MEMBERS = 3               # Number of ensemble members
SEED = 42
LEARNING_RATE = 1e-3        # Set to None to use default from config
# ============================================================

from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import EvalCallback

# Output directory
MODEL_DIR = os.path.join(PROJECT_ROOT, "RLM", "ensemble_dqn", "policy_weights")
os.makedirs(MODEL_DIR, exist_ok=True)

models = []

for i in range(N_MEMBERS):
    seed = SEED + i * 100
    print(f"\n{'='*40}")
    print(f"Training member {i + 1}/{N_MEMBERS} (seed={seed})")
    print(f"{'='*40}")

    # Create separate training env per member (different seed = different augmentation)
    train_env = TradingEnv(
        prices_df=prices_df, trades_df=trades_df,
        products=PRODUCTS, day=TRAIN_CONFIG["train_days"][0],
        augment=True, seed=seed,
    )
    eval_env_member = TradingEnv(
        prices_df=prices_df, trades_df=trades_df,
        products=PRODUCTS, day=TRAIN_CONFIG["eval_days"][0],
        augment=False, seed=seed + 1,
    )

    # Set normalization params
    for product in PRODUCTS:
        train_env.feature_computers[product].feature_means = feat_means
        train_env.feature_computers[product].feature_stds = feat_stds
        eval_env_member.feature_computers[product].feature_means = feat_means
        eval_env_member.feature_computers[product].feature_stds = feat_stds

    # Member output directory
    member_dir = os.path.join(MODEL_DIR, f"member_{i}")
    os.makedirs(member_dir, exist_ok=True)

    eval_callback = EvalCallback(
        eval_env_member,
        best_model_save_path=member_dir,
        log_path=member_dir,
        eval_freq=max(TOTAL_TIMESTEPS // 20, 1000),
        n_eval_episodes=5,
        deterministic=True,
        verbose=1,
    )

    model = DQN(
        "MlpPolicy",
        train_env,
        learning_rate=LEARNING_RATE or DQN_CONFIG["learning_rate"],
        buffer_size=DQN_CONFIG["buffer_size"],
        learning_starts=DQN_CONFIG["learning_starts"],
        batch_size=DQN_CONFIG["batch_size"],
        gamma=DQN_CONFIG["gamma"],
        exploration_fraction=DQN_CONFIG["exploration_fraction"],
        exploration_initial_eps=DQN_CONFIG["exploration_initial_eps"],
        exploration_final_eps=DQN_CONFIG["exploration_final_eps"],
        target_update_interval=DQN_CONFIG["target_update_interval"],
        train_freq=DQN_CONFIG["train_freq"],
        gradient_steps=DQN_CONFIG["gradient_steps"],
        policy_kwargs=dict(net_arch=NETWORK_CONFIG["hidden_sizes"]),
        device="auto",
        verbose=1,
        seed=seed,
    )

    print(f"Device: {model.device}")
    print(f"Training for {TOTAL_TIMESTEPS:,} timesteps...")

    model.learn(
        total_timesteps=TOTAL_TIMESTEPS,
        callback=eval_callback,
        progress_bar=True,
    )

    final_path = os.path.join(member_dir, "final_model")
    model.save(final_path)
    models.append(model)
    print(f"  Member {i + 1} saved to: {final_path}")

# Save normalization params
np.savez(os.path.join(MODEL_DIR, "norm_params.npz"),
         feature_means=feat_means, feature_stds=feat_stds)

print(f"\n{'='*60}")
print(f"Ensemble training complete! {N_MEMBERS} members trained.")
print(f"Models saved in: {MODEL_DIR}")

## 6. Evaluate Ensemble on Held-Out Day

Load all members and average Q-values for action selection.

In [ ]:
# Evaluate ensemble by averaging Q-values
import matplotlib.pyplot as plt

eval_env = TradingEnv(
    prices_df=prices_df, trades_df=trades_df,
    products=PRODUCTS, day=TRAIN_CONFIG["eval_days"][0],
    augment=False, seed=SEED + 1,
)
for product in PRODUCTS:
    eval_env.feature_computers[product].feature_means = feat_means
    eval_env.feature_computers[product].feature_stds = feat_stds

n_eval_episodes = 10
episode_pnls = []
episode_rewards = []

for ep in range(n_eval_episodes):
    obs, info = eval_env.reset()
    total_reward = 0
    done = False

    while not done:
        # Average Q-values across all ensemble members
        obs_tensor = torch.FloatTensor(obs).unsqueeze(0)
        all_q = []
        for m in models:
            with torch.no_grad():
                q = m.q_net(obs_tensor.to(m.device))
                all_q.append(q.cpu().numpy())
        avg_q = np.mean(all_q, axis=0)
        action = int(np.argmax(avg_q))

        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        done = terminated or truncated

    episode_pnls.append(info.get("pnl", total_reward))
    episode_rewards.append(total_reward)
    print(f"Episode {ep+1}: PnL={episode_pnls[-1]:.2f}, Positions={info.get('positions', {})}")

pnls = np.array(episode_pnls)
print(f"\n{'='*40}")
print(f"Mean PnL:  {pnls.mean():.2f}")
print(f"Std PnL:   {pnls.std():.2f}")
print(f"Min PnL:   {pnls.min():.2f}")
print(f"Max PnL:   {pnls.max():.2f}")
if pnls.std() > 0:
    print(f"Sharpe:    {pnls.mean() / pnls.std():.2f}")

## 7. Export Ensemble Weights to Numpy

Extract all members' weights into a single `.npz` file. Each member's weights are prefixed with m0_, m1_, m2_, etc.

In [ ]:
# Export all ensemble members to a single .npz file
weights = {}

for i, m in enumerate(models):
    print(f"\nExporting member {i}...")
    layer_idx = 0
    for name, param in m.q_net.named_parameters():
        tensor = param.detach().cpu().numpy()
        print(f"  m{i}.{name}: {tensor.shape}")
        if "weight" in name:
            weights[f"m{i}_W{layer_idx}"] = tensor
        elif "bias" in name:
            weights[f"m{i}_B{layer_idx}"] = tensor
            layer_idx += 1

# Include normalization params
weights["feature_means"] = feat_means
weights["feature_stds"] = feat_stds

# Save
export_path = os.path.join(MODEL_DIR, "exported_weights.npz")
np.savez(export_path, **weights)

total_params = sum(v.size for k, v in weights.items() if "feature" not in k)
file_size = os.path.getsize(export_path)
print(f"\nExported {N_MEMBERS} members, {len(weights)} arrays, {total_params:,} parameters")
print(f"File size: {file_size / 1024:.1f} KB")
print(f"Saved to: {export_path}")

# Quick verification: load and check member count
check = np.load(export_path)
member_keys = set()
for k in check.files:
    if k.startswith("m") and "_" in k:
        member_keys.add(k.split("_")[0])
print(f"\nVerification: found {len(member_keys)} members in exported file")

## 8. Download Weights

Run this cell to download the exported weights to your local machine. Then use `build_submission.py` locally to create the submission file.

In [ ]:
# Download/save weights
try:
    from google.colab import files
    files.download(export_path)
    print("Download started! Check your browser downloads.")
except ImportError:
    print("Not running on Colab. Weights saved to:", export_path)
except Exception as e:
    print(f"Colab download not available ({e}).")
    print(f"Weights saved to: {export_path}")
    print("Access them from the file browser or copy from the path above.")